# Study 931 — The CEF IPO Hole 🕳️

**A closed-end fund IPOs at $20. What is it worth the morning after?**

Not $20. The underwriting syndicate's 3-6% comes out of the proceeds, so the fund starts
life owning less than the subscriber paid — and unlike a company IPO there is no growth
story to reprice, because the fund is a basket of securities anyone can buy directly. The
folk claim: the price holds while the syndicate stabilises it, then slides to the discount
seasoned closed-end funds trade at.

We test it on **28 US closed-end funds that came to market between 2012 and
2022** (9 vintages — these are the large launches we could name and that
still trade under their IPO ticker, *not* every closed-end fund that ever came to market),
each raced against **one asset-class benchmark ETF**, on daily **total-return** closes
(2012-05-25 → 2026-06-30, 3,543 days), entering at the **offering price the
subscriber pays**.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `d69758d98b1a`); the
only live cells run the offline **synthetic** control and are labelled as such.
As-of 2026-06-30.*


## 1. What you actually buy for $20

You wire $20 a share. The underwriters keep roughly **$0.90** of it. The fund buys bonds or shares with the remaining **$19.10**. From the first minute, you own $19.10 of assets and a receipt saying you paid $20 — and there is no business inside that might grow into the difference. That is the whole idea this study measures.

> 🔬 **For the quants:** the load is quoted for context and is *never subtracted* anywhere in this study. It is the mechanism, not an extra cost — the price tape already contains its consequences, and subtracting it would double-count.

## 2. The month of quiet, then the fall

We compare each fund with an ETF holding the same *kind* of assets — a tech CEF against a tech ETF, a junk-bond CEF against a junk-bond ETF — so a fund launched into a bad year isn't blamed for the year.

In [1]:
R = dict(m1=-1.47, m1_t=-1.34, m3=-5.13, m3_t=-3.94, m6=-7.06, m6_t=-4.06, m12=-10.26, m12_t=-3.65,
         m12_neg=0.89, n_funds=28)
for lab in ('1', '3', '6', '12'):
    print('%3s months after the IPO: %+6.2f%% vs its own asset class   (t = %+.2f)'
          % (lab, R['m'+lab], R['m'+lab+'_t']))
print()
print('%.0f%% of the %d funds are behind their asset class one year in.'
      % (R['m12_neg']*100, R['n_funds']))

  1 months after the IPO:  -1.47% vs its own asset class   (t = -1.34)
  3 months after the IPO:  -5.13% vs its own asset class   (t = -3.94)
  6 months after the IPO:  -7.06% vs its own asset class   (t = -4.06)
 12 months after the IPO: -10.26% vs its own asset class   (t = -3.65)

89% of the 28 funds are behind their asset class one year in.


Month one is **quiet** — -1.47%, which is statistical nothing (*t* = -1.34). That is the syndicate holding the price up. Then the support ends and the hole opens: **-5.13%** by month three, **-10.26%** by month twelve. Twenty-five of twenty-eight funds are behind. This is not a bad year for one sleeve — it happened to tech funds, health funds, loan funds, preferred funds and credit funds alike, in **9 different vintage years**.

## 3. Seven weeks to a discount

We cannot watch the discount itself — daily net asset values aren't on the free tape — so we watch the next best thing: the day the fund first falls a given distance behind the assets it holds. **27 of 28** funds fall 5% behind within two years, and the median takes **34 trading days** — about seven weeks from the closing bell of the IPO. Ten percent behind takes a median 97 days.

> 🔬 **For the quants:** a PROXY for time-to-discount, not the discount. It conflates the price/NAV gap with any tracking difference between the fund's portfolio and the benchmark ETF — which is why the *level* is indicative and only the *timing* pattern is taken seriously.

## 4. Is it the IPO, or are these just bad funds?

The obvious objection. So we ran the identical measurement on the identical funds from **fake** start dates drawn at random from their seasoned life. The average abnormal return over those fake windows is **+1.90%** (sd 2.30%) — nothing like -10.26%. And measured from their first birthday to their third, the same funds show **+3.02%** (*t* = +0.83): the bleeding simply stops.

The damage lives **in the IPO window**. These are not permanently bad assets — they are permanently bad *purchases at issue*.

## 5. So can you make money from it?

The trade you would want is to short the new fund and buy the benchmark. On paper it pays **+9.08%** over twelve months. In practice you have to borrow the shares, and the trade breaks even at about **908 bps a year** of borrow cost — while a closed-end fund in the weeks after its IPO is the hardest thing on the market to borrow: the whole issue is sitting in the retail accounts the syndicate placed it in, and almost none of it is lendable.

There is no long version either. You cannot buy a hole. What is left is one instruction, and it is free: **never subscribe to a closed-end fund IPO — wait a year and buy it in the market**, most likely at a discount, from the person who did subscribe.

## 6. Live check — is the measuring stick straight? *(synthetic, not the real tape)*

Before believing a −10% result, check that the estimator finds a hole when one is planted and finds nothing when none is. Both worlds below are **simulated** — no real fund appears in this cell.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from cef_ipo import data, strategy as st
planted, truth = data.synthetic_panel(signal_strength=1.0, seed=931)
null, _ = data.synthetic_panel(signal_strength=0.0, seed=931)
p = st.synthetic_detect(planted)
n = st.synthetic_detect(null)
print('SYNTHETIC world with a planted %+.0f%% first-year hole -> measured %+.2f%% (t = %+.2f)'
      % (-truth['planted_slide']*100, p['mean_abn_pct'], p['tstat']))
print('SYNTHETIC world with no hole at all               -> measured %+.2f%% (t = %+.2f)'
      % (n['mean_abn_pct'], n['tstat']))

SYNTHETIC world with a planted -10% first-year hole -> measured -9.70% (t = -6.70)
SYNTHETIC world with no hole at all               -> measured +0.15% (t = +0.10)


## Verdict

- **Signal — Real.** New closed-end funds lose ground to their own asset class: **-5.13%** by month three (*t* = **-3.94**) and about **-10.26%** by month twelve, in both vintage halves, under every benchmark choice, not because the funds are leveraged, and only inside the IPO window. Honest caveats: at twelve months the most conservative test does not clear (so treat 10% as the size, three months as the proof); the 28 names are the big launches we could name, not every closed-end fund that ever IPO'd; and they are funds that still exist under their IPO ticker (the worst ones get merged away, which makes the hole look *smaller*).
- **Tradability — Mirage.** The short that would monetise it cannot be borrowed at a price that leaves anything (break-even ~908 bps/yr of borrow). The finding is worth exactly one behaviour — wait — and behaviour doesn't pay a coupon.